# NSMC 감성분석 프로젝트: klue/bert-base fine-tuning

이번 프로젝트에서는 한국어 영화 리뷰 데이터셋인 NSMC를 사용해서 감성분석 모델을 학습합니다.

실습에서 배운 Hugging Face 흐름을 그대로 가져오되, 데이터셋과 모델을 한국어 프로젝트에 맞게 바꿉니다.

```text
MRPC + distilbert-base-uncased
-> NSMC + klue/bert-base
```

루브릭 기준은 다음 세 가지입니다.

1. `klue/bert-base` 모델과 NSMC 데이터를 정상적으로 불러오고 fine-tuning이 작동하는지 확인한다.
2. 전처리와 fine-tuning을 개선해서 validation accuracy 90% 이상을 목표로 한다.
3. bucketing을 적용한 학습 결과와 적용하지 않은 결과를 비교한다.

## 프로젝트 진행 계획

실습 노트북에서 배운 순서를 프로젝트에 맞게 다시 진행합니다.

1. 라이브러리와 GPU 환경 확인
2. NSMC 데이터셋 불러오기
3. `klue/bert-base` tokenizer 불러오기
4. NSMC 문장 토큰화
5. 학습용 데이터셋 정리
6. `klue/bert-base` 분류 모델 불러오기
7. Trainer로 baseline 학습
8. accuracy 90% 이상을 목표로 학습 설정 개선
9. bucketing 적용 후 결과 비교

## 최적화 아이디어 메모

퍼실 힌트로 받은 성능/학습 최적화 후보입니다. 기본 fine-tuning이 먼저 정상 작동한 뒤 비교 실험 후보로 사용합니다.

- **Layer Normalization**
  - 딥러닝 모델 내부의 값 분포를 안정화해서 학습을 더 잘 되게 만드는 기법입니다.
  - BERT 계열 모델에는 이미 Layer Normalization이 구조 안에 포함되어 있습니다.
  - 프로젝트에서는 직접 새로 구현하기보다, 모델 구조를 이해하거나 학습 안정화 관점에서 확인할 포인트로 둡니다.

- **QLoRA**
  - 큰 모델을 적은 GPU 메모리로 fine-tuning하기 위한 방법입니다.
  - 모델 전체를 다 학습하지 않고, 일부 작은 어댑터 파라미터만 효율적으로 학습합니다.
  - NSMC 기본 목표를 달성한 뒤, 메모리 절약이나 추가 실험 후보로 검토합니다.

## STEP 0. 라이브러리와 GPU 환경 확인

먼저 프로젝트에 필요한 라이브러리가 잘 불러와지는지 확인합니다.

이번 프로젝트에서는 `transformers`, `datasets`, `torch`가 핵심입니다.

In [40]:
import numpy as np
import pandas as pd
import torch
import transformers
import datasets

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('cuda 사용 가능:', torch.cuda.is_available())

numpy: 2.4.5
pandas: 3.0.3
torch: 2.11.0+cu128
transformers: 5.8.1
datasets: 4.8.5
cuda 사용 가능: True


## STEP 1. NSMC 데이터셋 불러오기

NSMC는 네이버 영화 리뷰 문장과 긍정/부정 라벨로 구성된 한국어 감성분석 데이터셋입니다.

라벨은 보통 다음과 같이 해석합니다.

```text
0 -> 부정
1 -> 긍정
```

먼저 Hugging Face `datasets` 라이브러리로 데이터를 불러오고, train/test split과 컬럼 구조를 확인합니다.

In [41]:
from datasets import load_dataset

# 현재 datasets 버전에서는 load_dataset('nsmc')가 예전 dataset script 문제로 실패할 수 있습니다.
# 그래서 NSMC 원본 GitHub의 TSV 파일을 csv 로더로 직접 불러옵니다.
data_files = {
    'train': 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt',
    'test': 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt',
}

nsmc = load_dataset('csv', data_files=data_files, sep='\t')

nsmc

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})

In [42]:
# 데이터의 컬럼과 feature 정보를 확인합니다.
# document: 리뷰 문장
# label: 감성 라벨, 0은 부정 / 1은 긍정

print(nsmc['train'].column_names)
print(nsmc['train'].features)
print(nsmc['train'][0])

['id', 'document', 'label']
{'id': Value('int64'), 'document': Value('large_string'), 'label': Value('int64')}
{'id': 9976970, 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}


In [43]:
# train/test 데이터 개수를 확인합니다.
# 학습에는 train을 사용하고, 최종 성능 확인에는 test를 사용할 예정입니다.

print('train 개수:', len(nsmc['train']))
print('test 개수:', len(nsmc['test']))

train 개수: 150000
test 개수: 50000


## STEP 2. klue/bert-base tokenizer 불러오기

실습에서는 영어 모델인 `distilbert-base-uncased` tokenizer를 사용했습니다.

이번 프로젝트에서는 한국어 BERT 모델인 `klue/bert-base`에 맞는 tokenizer를 사용합니다.

NSMC는 리뷰 문장 하나를 보고 긍정/부정을 분류하는 문제이므로, tokenizer에는 `document` 컬럼 하나만 넣습니다.

In [44]:
from transformers import AutoTokenizer

model_name = 'klue/bert-base'

# 모델 이름에 맞는 tokenizer를 자동으로 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer

BertTokenizer(name_or_path='klue/bert-base', vocab_size=32000, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [56]:
# NSMC 샘플 하나를 확인합니다.

sample = nsmc['train'][0]

print('document:', sample['document'])
print('label:', sample['label'])

document: 아 더빙.. 진짜 짜증나네요 목소리
label: 0


In [46]:
# 리뷰 문장 하나를 tokenizer에 넣어봅니다.
# truncation=True는 문장이 너무 길면 모델 입력 길이에 맞게 자르겠다는 뜻입니다.

encoded = tokenizer(sample['document'], truncation=True)

encoded

{'input_ids': [2, 1376, 831, 2604, 18, 18, 4229, 9801, 2075, 2203, 2182, 4243, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [57]:
# input_ids를 다시 token 문자열로 바꿔서 확인합니다.
# 한국어 BERT tokenizer가 문장을 어떤 토큰 조각으로 나누는지 볼 수 있습니다.

tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])

print(tokens)

['[CLS]', '아', '더', '##빙', '.', '.', '진짜', '짜증', '##나', '##네', '##요', '목소리', '[SEP]']


### STEP 2 정리

이번 단계에서는 `klue/bert-base` tokenizer를 불러오고, NSMC 리뷰 문장 하나를 숫자로 바꿔봤습니다.

실습과의 차이는 다음과 같습니다.

- MRPC: `sentence1`, `sentence2` 두 문장을 함께 tokenizer에 넣음
- NSMC: `document` 한 문장만 tokenizer에 넣음

다음 단계에서는 NSMC 전체 데이터셋에 tokenizer를 적용합니다.

## STEP 3. NSMC 전체 데이터셋에 tokenizer 적용하기

샘플 하나가 잘 토큰화되는 것을 확인했으므로, 이제 train/test 전체 데이터셋에 tokenizer를 적용합니다.

NSMC는 문장 하나를 분류하는 문제라서 `document` 컬럼만 tokenizer에 넣습니다.

또한 원본 데이터에는 간혹 비어 있는 리뷰가 있을 수 있으므로, 먼저 `document`가 없는 샘플을 제거합니다.

In [58]:
# document가 None이거나 빈 문자열인 샘플을 제거합니다.
# BERT tokenizer는 실제 문장이 있어야 안정적으로 동작합니다.

def has_document(example):
    return example['document'] is not None and len(example['document'].strip()) > 0

nsmc_clean = nsmc.filter(has_document)

print('원본 train 개수:', len(nsmc['train']))
print('정리 후 train 개수:', len(nsmc_clean['train']))
print('원본 test 개수:', len(nsmc['test']))
print('정리 후 test 개수:', len(nsmc_clean['test']))

원본 train 개수: 150000
정리 후 train 개수: 149995
원본 test 개수: 50000
정리 후 test 개수: 49997


In [59]:
# NSMC 문장을 tokenizer에 넣는 함수를 만듭니다.
# padding은 여기서 고정하지 않고, 나중에 DataCollatorWithPadding으로 batch마다 동적으로 처리합니다.

def tokenize_nsmc(batch):
    return tokenizer(
        batch['document'],
        truncation=True,
        max_length=128
    )

In [60]:
# map을 사용해서 train/test 전체에 tokenizer를 적용합니다.
# batched=True는 여러 샘플을 묶어서 처리하므로 더 빠릅니다.

tokenized_nsmc = nsmc_clean.map(tokenize_nsmc, batched=True)

tokenized_nsmc

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 149995
    })
    test: Dataset({
        features: ['id', 'document', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 49997
    })
})

In [61]:
# tokenizer 적용 후 컬럼을 확인합니다.
# 기존 id, document, label에 input_ids, token_type_ids, attention_mask가 추가됩니다.

print(tokenized_nsmc['train'].column_names)
tokenized_nsmc['train'][0]

['id', 'document', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


{'id': 9976970,
 'document': '아 더빙.. 진짜 짜증나네요 목소리',
 'label': 0,
 'input_ids': [2,
  1376,
  831,
  2604,
  18,
  18,
  4229,
  9801,
  2075,
  2203,
  2182,
  4243,
  3],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

### STEP 3 정리

이번 단계에서는 NSMC 전체 데이터셋에 tokenizer를 적용했습니다.

- `document`가 비어 있는 샘플 제거
- `document` 컬럼을 tokenizer에 입력
- `input_ids`, `token_type_ids`, `attention_mask` 컬럼 생성

다음 단계에서는 학습에 필요하지 않은 원문 컬럼을 제거하고, train 데이터에서 validation 데이터를 나눕니다.

## STEP 4. 학습용 데이터셋 정리 및 validation 분리

NSMC 원본 데이터는 train/test만 제공합니다.

하지만 학습 중 성능을 확인하려면 validation 데이터가 필요합니다.

따라서 train 데이터 일부를 validation으로 나누고, 학습에 직접 필요하지 않은 원문 컬럼을 제거합니다.

In [62]:
# 학습에 직접 쓰지 않을 컬럼을 제거합니다.
# document는 이미 input_ids로 변환되었고, id는 학습에 필요하지 않습니다.

columns_to_remove = ['id', 'document']

tokenized_nsmc_for_train = tokenized_nsmc.remove_columns(columns_to_remove)

tokenized_nsmc_for_train

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 149995
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 49997
    })
})

In [63]:
# train 데이터에서 validation 데이터를 분리합니다.
# stratify_by_column='label'을 사용하면 긍정/부정 비율을 비슷하게 유지할 수 있습니다.
# 단, stratify를 쓰려면 label 컬럼이 ClassLabel 타입이어야 합니다.

from datasets import ClassLabel

label_feature = ClassLabel(names=['negative', 'positive'])
tokenized_nsmc_for_train = tokenized_nsmc_for_train.cast_column('label', label_feature)

split_dataset = tokenized_nsmc_for_train['train'].train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column='label'
)

train_dataset = split_dataset['train']
valid_dataset = split_dataset['test']
test_dataset = tokenized_nsmc_for_train['test']

print('train:', len(train_dataset))
print('valid:', len(valid_dataset))
print('test:', len(test_dataset))

train: 134995
valid: 15000
test: 49997


In [64]:
# 정리된 학습 데이터 하나를 확인합니다.
# 이제 모델 입력과 label만 남아 있어야 합니다.

train_dataset[0]

{'label': 0,
 'input_ids': [2,
  4174,
  26823,
  2115,
  2477,
  2499,
  5,
  10133,
  2116,
  2227,
  2209,
  2155,
  2075,
  5,
  5,
  28816,
  4847,
  2208,
  11990,
  2116,
  2227,
  2961,
  2062,
  3],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [65]:
# label 분포를 간단히 확인합니다.
# 0과 1의 비율이 크게 치우치지 않았는지 봅니다.

from collections import Counter

print('train label:', Counter(train_dataset['label']))
print('valid label:', Counter(valid_dataset['label']))
print('test label:', Counter(test_dataset['label']))

train label: Counter({0: 67653, 1: 67342})
valid label: Counter({0: 7517, 1: 7483})
test label: Counter({1: 25171, 0: 24826})


### STEP 4 정리

이번 단계에서는 학습에 사용할 데이터셋을 정리했습니다.

- 제거한 컬럼: `id`, `document`
- 남긴 컬럼: `label`, `input_ids`, `token_type_ids`, `attention_mask`
- train 데이터 일부를 validation 데이터로 분리

다음 단계에서는 `klue/bert-base` 분류 모델을 불러옵니다.

## STEP 5. klue/bert-base 분류 모델 불러오기

이제 NSMC 리뷰를 긍정/부정으로 분류할 모델을 불러옵니다.

실습에서 사용한 구조와 같습니다.

```text
BERT 본체 + classification head
```

`klue/bert-base` 본체는 한국어 문장을 이해하도록 사전학습된 모델이고, classification head는 NSMC의 0/1 라벨을 예측하기 위해 새로 붙는 부분입니다.

In [66]:
from transformers import AutoModelForSequenceClassification

# NSMC는 부정/긍정 2개 라벨을 예측하는 이진 분류 문제입니다.
id2label = {0: 'negative', 1: 'positive'}
label2id = {'negative': 0, 'positive': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2439.44it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
# 모델 설정을 확인합니다.

print('model name:', model_name)
print('num_labels:', model.config.num_labels)
print('id2label:', model.config.id2label)
print('label2id:', model.config.label2id)

model name: klue/bert-base
num_labels: 2
id2label: {0: 'negative', 1: 'positive'}
label2id: {'negative': 0, 'positive': 1}


### STEP 5 정리

이번 단계에서는 `klue/bert-base`를 문장 분류용 모델로 불러왔습니다.

처음 불러올 때 classifier weight가 새로 초기화되었다는 경고가 나올 수 있습니다.

이것은 정상입니다. 아직 NSMC를 학습하지 않은 classification head가 새로 붙었기 때문입니다.

다음 단계에서는 Trainer로 baseline fine-tuning을 진행합니다.

## STEP 6. Baseline fine-tuning

이제 `Trainer`를 사용해서 baseline 학습을 진행합니다.

이번 단계의 목표는 최고 성능을 바로 내는 것이 아니라, 아래 루브릭의 첫 번째 조건을 만족하는 것입니다.

```text
klue/bert-base를 NSMC 데이터셋으로 fine-tuning하여 모델이 정상적으로 작동하는 것을 확인한다.
```

성능 개선은 baseline 결과를 확인한 뒤 다음 단계에서 진행합니다.

### Baseline 설정 메모

처음에는 `per_device_train_batch_size=16`으로 설정했지만, NSMC 데이터셋 크기 때문에 학습 시간이 너무 길었다.

따라서 GPU 메모리 여유를 고려해 baseline의 batch size를 조정했다.

- train batch size: 16 -> 32
- eval batch size: 16 -> 64
- mixed precision: GPU가 bf16을 지원하면 bf16 사용, 아니면 fp16 사용

이 조정은 성능 개선 실험이라기보다, baseline 학습을 현실적으로 실행하기 위한 환경 최적화이다.

In [67]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

# batch마다 가장 긴 문장 길이에 맞춰 padding합니다.
# 고정 길이 padding보다 불필요한 계산을 줄일 수 있습니다.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# GPU가 있으면 fp16을 사용해 학습 속도와 메모리 사용량을 개선합니다.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

In [68]:
# accuracy를 계산하는 함수입니다.
# predictions는 각 label에 대한 점수이므로 argmax로 가장 높은 점수의 label을 고릅니다.

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predicted_labels = np.argmax(predictions, axis=1)
    accuracy = (predicted_labels == labels).mean()
    return {'accuracy': accuracy}

In [69]:
# baseline 학습 설정입니다.
# 처음에는 안정적인 기본 설정으로 모델이 정상 작동하는지 확인합니다.

training_args = TrainingArguments(
    output_dir='./nsmc_klue_baseline',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_strategy='steps',
    logging_steps=200,
    report_to='none',
    seed=42
)

In [70]:
# Trainer를 구성합니다.
# transformers 버전에 따라 tokenizer 인자 대신 processing_class를 사용합니다.

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

baseline_train_result = trainer.train()
baseline_train_result

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# baseline validation 성능을 확인합니다.

baseline_valid_result = trainer.evaluate(valid_dataset)
baseline_valid_result

Training Loss,Validation Loss,Epoch,Accuracy
0.112633,0.285712,2,0.910400


{'eval_loss': 0.2857121229171753, 'eval_accuracy': 0.9104}

In [ ]:
# 최종 test 성능도 확인합니다.
# test 데이터는 모델 선택에 사용하지 않고, 최종 확인용으로만 봅니다.

baseline_test_result = trainer.evaluate(test_dataset)
baseline_test_result

Training Loss,Validation Loss,Epoch,Accuracy
0.112633,0.299156,2,0.905354


{'eval_loss': 0.2991563081741333, 'eval_accuracy': 0.9053543212592755}

### STEP 6 정리

이번 단계에서는 baseline fine-tuning을 수행합니다.

확인할 값은 다음과 같습니다.

- `eval_loss`
- `eval_accuracy`
- 학습이 정상적으로 끝났는지 여부

다음 단계에서는 baseline 결과를 바탕으로 validation accuracy 90% 이상을 목표로 학습 설정을 개선합니다.

## STEP 7. Bucketing 적용 학습

baseline에서 validation accuracy가 90% 이상 나왔으므로, 이제 bucketing을 적용한 학습을 진행합니다.

현재 baseline은 `DataCollatorWithPadding`을 사용하므로 batch 단위 dynamic padding은 이미 적용되어 있습니다.

이번 단계에서는 여기에 `group_by_length=True`를 추가합니다.

`group_by_length=True`는 길이가 비슷한 샘플끼리 batch를 구성해서 padding 낭비를 줄이는 설정입니다.

공정한 비교를 위해 baseline에서 학습된 모델을 재사용하지 않고, 같은 `klue/bert-base` 모델을 새로 불러와 다시 학습합니다.

In [ ]:
# baseline 결과를 비교용으로 기록합니다.
# 앞 단계에서 baseline_valid_result와 baseline_test_result가 생성되어 있어야 합니다.

baseline_valid_result

{'eval_loss': 0.2857121229171753, 'eval_accuracy': 0.9104}

In [ ]:
# 공정 비교를 위해 모델을 새로 불러옵니다.
# 기존 model은 baseline 학습으로 이미 weight가 업데이트된 상태입니다.

bucket_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3154.54it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [ ]:
# bucketing 적용 학습 설정입니다.
# baseline 설정과 거의 같고, train_sampling_strategy='group_by_length'만 추가합니다.

training_args_bucket = TrainingArguments(
    output_dir='./nsmc_klue_bucket',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_strategy='steps',
    logging_steps=200,
    report_to='none',
    seed=42,
    train_sampling_strategy='group_by_length'
)

In [ ]:
bucket_trainer = Trainer(
    model=bucket_model,
    args=training_args_bucket,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

bucket_train_result = bucket_trainer.train()
bucket_train_result

Epoch,Training Loss,Validation Loss,Accuracy
1,0.238037,0.226608,0.908133
2,0.175051,0.234874,0.911933


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]


TrainOutput(global_step=8438, training_loss=0.2290444082489339, metrics={'train_runtime': 768.4681, 'train_samples_per_second': 351.335, 'train_steps_per_second': 10.98, 'total_flos': 3254929393640040.0, 'train_loss': 0.2290444082489339, 'epoch': 2.0})

In [ ]:
# bucketing 적용 모델의 validation 성능을 확인합니다.

bucket_valid_result = bucket_trainer.evaluate(valid_dataset)
bucket_valid_result

Training Loss,Validation Loss,Epoch,Accuracy
0.175051,0.234884,2,0.912000


{'eval_loss': 0.2348838895559311, 'eval_accuracy': 0.912}

In [ ]:
# bucketing 적용 모델의 test 성능도 확인합니다.

bucket_test_result = bucket_trainer.evaluate(test_dataset)
bucket_test_result

Training Loss,Validation Loss,Epoch,Accuracy
0.175051,0.252519,2,0.904854


{'eval_loss': 0.2525193393230438, 'eval_accuracy': 0.9048542912574754}

### STEP 7 정리

이번 단계에서는 `group_by_length=True`로 bucketing을 적용했습니다.

비교할 값은 다음과 같습니다.

- baseline validation accuracy
- bucketing validation accuracy
- baseline 학습 시간
- bucketing 학습 시간

다음 단계에서는 두 결과를 표로 정리하고, 성능과 학습 시간의 trade-off를 분석합니다.

## STEP 8. Baseline과 Bucketing 결과 비교

마지막으로 baseline 학습 결과와 bucketing 적용 결과를 비교합니다.

루브릭에서는 bucketing 적용 여부와 fine-tuning 연산 속도, 모델 성능 사이의 trade-off를 확인하고 분석하는 것을 요구합니다.

따라서 accuracy뿐 아니라 학습 시간도 함께 비교합니다.

In [ ]:
# Trainer의 train 결과에서 runtime 정보를 꺼냅니다.
# baseline_train_result는 trainer.train()의 반환값을 저장해둔 경우 사용할 수 있습니다.
# 만약 앞 단계에서 저장하지 않았다면, 출력 로그의 train_runtime 값을 직접 기록해도 됩니다.

baseline_train_metrics = getattr(globals().get('baseline_train_result', None), 'metrics', {})
bucket_train_metrics = getattr(globals().get('bucket_train_result', None), 'metrics', {})

# baseline을 trainer.train()으로만 실행해서 변수에 저장하지 못한 경우,
# 노트북 출력에 남아 있던 TrainOutput 값을 수동으로 복원합니다.
if not baseline_train_metrics:
    baseline_train_metrics = {
        'train_runtime': 1164.9011,
        'train_samples_per_second': 231.771,
        'train_steps_per_second': 7.244,
        'train_loss': 0.1574057102429412,
        'epoch': 2.0,
    }

baseline_train_metrics, bucket_train_metrics

({'train_runtime': 1164.9011,
  'train_samples_per_second': 231.771,
  'train_steps_per_second': 7.244,
  'train_loss': 0.1574057102429412,
  'epoch': 2.0},
 {'train_runtime': 768.4681,
  'train_samples_per_second': 351.335,
  'train_steps_per_second': 10.98,
  'total_flos': 3254929393640040.0,
  'train_loss': 0.2290444082489339,
  'epoch': 2.0})

In [ ]:
# 결과 비교 표를 만듭니다.
# 값이 없는 항목은 None으로 표시됩니다.

comparison = pd.DataFrame([
    {
        'experiment': 'baseline_dynamic_padding',
        'validation_accuracy': baseline_valid_result.get('eval_accuracy'),
        'test_accuracy': baseline_test_result.get('eval_accuracy'),
        'train_runtime_sec': baseline_train_metrics.get('train_runtime'),
        'samples_per_second': baseline_train_metrics.get('train_samples_per_second'),
        'group_by_length': False,
        'learning_rate': 2e-5,
        'warmup_ratio': 0.0,
    },
    {
        'experiment': 'bucketing_group_by_length',
        'validation_accuracy': bucket_valid_result.get('eval_accuracy'),
        'test_accuracy': bucket_test_result.get('eval_accuracy'),
        'train_runtime_sec': bucket_train_metrics.get('train_runtime'),
        'samples_per_second': bucket_train_metrics.get('train_samples_per_second'),
        'group_by_length': True,
        'learning_rate': 2e-5,
        'warmup_ratio': 0.0,
    },
])

comparison

,experiment,validation_accuracy,test_accuracy,train_runtime_sec,samples_per_second,group_by_length,learning_rate,warmup_ratio
0,baseline_dynamic_padding,0.9104,0.905354,1164.9011,231.771,False,0.00002,0.0
1,bucketing_group_by_length,0.9120,0.904854,768.4681,351.335,True,0.00002,0.0


In [ ]:
# 결과를 csv로 저장해두면 나중에 회고나 제출 정리에 활용하기 좋습니다.

comparison.to_csv('nsmc_klue_result_comparison.csv', index=False)
print('saved: nsmc_klue_result_comparison.csv')

saved: nsmc_klue_result_comparison.csv


### 결과 분석 메모

아래 항목을 실제 실행 결과에 맞게 채웁니다.

- Baseline validation accuracy: `0.9104`
- Bucketing validation accuracy: `0.9120`
- Baseline test accuracy: `0.9053`
- Bucketing test accuracy: `0.9048`
- Baseline train runtime: `1164.9011`
- Bucketing train runtime: `768.4681`

분석 방향:

1. Baseline만으로 validation accuracy 90% 이상을 달성했는지 확인한다.
2. Bucketing 적용 후 validation accuracy와 test accuracy가 유지되거나 개선되었는지 확인한다.
3. Bucketing 적용 후 학습 시간이 줄었는지, 또는 samples/sec가 증가했는지 확인한다.
4. validation accuracy와 test accuracy가 서로 다르게 움직이면, validation 기준 최적화와 test 일반화 성능 사이의 차이를 언급한다.
5. 최종 모델은 accuracy뿐 아니라 학습 시간, test 성능, 재현 가능성을 함께 고려해 선택한다.

## STEP 9. 추가 실험

baseline과 bucketing 실험 이후, 성능 향상 가능성을 확인하기 위해 추가 실험을 진행합니다.

추가 실험은 기존 결과와 공정하게 비교하기 위해 매번 `klue/bert-base` 모델을 새로 불러와 학습합니다.

### STEP 9-1. learning rate 낮추기

baseline과 bucketing 결과에서 validation accuracy는 90%를 넘겼지만, validation loss가 epoch가 진행되며 증가하는 경향이 있었습니다.

따라서 learning rate를 `2e-5`에서 `1e-5`로 낮춰 더 천천히 학습하는 실험을 진행합니다.

비교를 공정하게 하기 위해 bucketing 모델을 이어서 학습하지 않고, `klue/bert-base` 모델을 새로 불러와 학습합니다.

이 실험은 다음 조건을 사용합니다.

- learning rate: `1e-5`
- bucketing: 적용
- epoch: 2
- batch size: baseline/bucketing 실험과 동일

In [ ]:
# learning rate 추가 실험을 위해 모델을 새로 불러옵니다.

lr_experiment_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3257.37it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [ ]:
# learning rate를 1e-5로 낮춘 추가 실험 설정입니다.
# bucketing 설정은 유지합니다.

training_args_lr = TrainingArguments(
    output_dir='./nsmc_klue_lr1e5_bucket',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_strategy='steps',
    logging_steps=200,
    report_to='none',
    seed=42,
    train_sampling_strategy='group_by_length'
)

In [ ]:
lr_trainer = Trainer(
    model=lr_experiment_model,
    args=training_args_lr,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

lr_train_result = lr_trainer.train()
lr_train_result

Epoch,Training Loss,Validation Loss,Accuracy
1,0.250513,0.236880,0.902067
2,0.201875,0.235003,0.906600


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]


TrainOutput(global_step=8438, training_loss=0.2467365477372753, metrics={'train_runtime': 775.0499, 'train_samples_per_second': 348.352, 'train_steps_per_second': 10.887, 'total_flos': 3254929393640040.0, 'train_loss': 0.2467365477372753, 'epoch': 2.0})

In [ ]:
# learning rate 추가 실험의 validation 성능을 확인합니다.

lr_valid_result = lr_trainer.evaluate(valid_dataset)
lr_valid_result

Training Loss,Validation Loss,Epoch,Accuracy
0.201875,0.234992,2,0.906533


{'eval_loss': 0.23499244451522827, 'eval_accuracy': 0.9065333333333333}

In [ ]:
# learning rate 추가 실험의 test 성능을 확인합니다.

lr_test_result = lr_trainer.evaluate(test_dataset)
lr_test_result

Training Loss,Validation Loss,Epoch,Accuracy
0.201875,0.251457,2,0.901434


{'eval_loss': 0.2514568567276001, 'eval_accuracy': 0.9014340860451627}

### STEP 9-1 정리

이번 단계에서는 learning rate를 낮춘 추가 실험을 진행합니다.

확인할 포인트는 다음과 같습니다.

- validation accuracy가 기존 bucketing 결과보다 좋아졌는지
- validation loss가 더 안정적인지
- 학습 시간이 크게 늘어나지 않았는지

성능이 좋아지면 최종 후보 모델로 고려하고, 좋아지지 않더라도 learning rate 변경 실험 결과로 분석에 포함할 수 있습니다.

### STEP 9-2. warmup scheduler 적용

learning rate를 `1e-5`로 낮춘 실험 A가 기대만큼 성능을 올리지 못했다면, 이번에는 learning rate 자체는 `2e-5`로 유지하고 scheduler를 추가합니다.

`warmup_ratio=0.1`은 전체 학습 step의 앞 10% 동안 learning rate를 서서히 올린 뒤, 이후에는 scheduler에 따라 줄이도록 합니다.

이 실험의 목적은 학습 초반을 안정화해서 validation/test 성능이 좋아지는지 확인하는 것입니다.

조건은 다음과 같습니다.

- learning rate: `2e-5`
- scheduler: `linear`
- warmup ratio: `0.1`
- bucketing: 적용
- epoch: 2

In [ ]:
# warmup scheduler 실험을 위해 모델을 새로 불러옵니다.

warmup_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3406.95it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [ ]:
# learning rate는 2e-5로 유지하고 warmup + linear scheduler를 추가합니다.

training_args_warmup = TrainingArguments(
    output_dir='./nsmc_klue_warmup_bucket',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_strategy='steps',
    logging_steps=200,
    report_to='none',
    seed=42,
    train_sampling_strategy='group_by_length'
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
warmup_trainer = Trainer(
    model=warmup_model,
    args=training_args_warmup,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

warmup_train_result = warmup_trainer.train()
warmup_train_result

Epoch,Training Loss,Validation Loss,Accuracy
1,0.242689,0.228878,0.904000
2,0.182076,0.231743,0.911733


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


TrainOutput(global_step=8438, training_loss=0.2414768259229748, metrics={'train_runtime': 764.0451, 'train_samples_per_second': 353.369, 'train_steps_per_second': 11.044, 'total_flos': 3254929393640040.0, 'train_loss': 0.2414768259229748, 'epoch': 2.0})

In [ ]:
# warmup scheduler 실험의 validation 성능을 확인합니다.

warmup_valid_result = warmup_trainer.evaluate(valid_dataset)
warmup_valid_result

Training Loss,Validation Loss,Epoch,Accuracy
0.182076,0.231732,2,0.911800


{'eval_loss': 0.2317322939634323, 'eval_accuracy': 0.9118}

In [ ]:
# warmup scheduler 실험의 test 성능을 확인합니다.

warmup_test_result = warmup_trainer.evaluate(test_dataset)
warmup_test_result

Training Loss,Validation Loss,Epoch,Accuracy
0.182076,0.248439,2,0.906114


{'eval_loss': 0.24843865633010864, 'eval_accuracy': 0.9061143668620117}

### STEP 9-2 정리

이번 단계에서는 learning rate를 낮추는 대신 warmup scheduler를 적용했습니다.

확인할 포인트는 다음과 같습니다.

- 기존 bucketing 결과보다 validation/test accuracy가 좋아졌는지
- validation loss가 더 안정적인지
- warmup으로 인해 학습 시간이 크게 증가하지 않았는지

만약 성능이 개선되지 않더라도, scheduler 적용이 항상 성능 향상으로 이어지지는 않는다는 실험 결과로 기록할 수 있습니다.

### STEP 9-3. Test accuracy 향상 실험: warmup + weight decay + early stopping

이번 실험은 test accuracy 0.91 이상을 목표로 합니다.

앞선 warmup scheduler 실험에서 validation loss가 안정화되는 경향을 보였으므로, 최대 epoch를 5로 늘리고 early stopping을 적용합니다.

또한 일반화 성능을 높이기 위해 `weight_decay`를 `0.01`에서 `0.05`로 올립니다.

조건은 다음과 같습니다.

- learning rate: `2e-5`
- warmup ratio: `0.1`
- scheduler: `linear`
- weight decay: `0.05`
- max epoch: `5`
- early stopping patience: `2`
- bucketing: 적용

In [71]:
from transformers import EarlyStoppingCallback

# test accuracy 향상 실험을 위해 모델을 새로 불러옵니다.
final_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4834.58it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [72]:
# 최대 5 epoch까지 학습하되, validation accuracy가 개선되지 않으면 early stopping으로 중단합니다.

training_args_final = TrainingArguments(
    output_dir='./nsmc_klue_final_warmup_wd005',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_strategy='steps',
    logging_steps=200,
    report_to='none',
    seed=42,
    train_sampling_strategy='group_by_length'
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [73]:
final_trainer = Trainer(
    model=final_model,
    args=training_args_final,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

final_train_result = final_trainer.train()
final_train_result

Epoch,Training Loss,Validation Loss,Accuracy
1,0.254869,0.240747,0.899733
2,0.201698,0.234173,0.911733
3,0.144394,0.263825,0.908400
4,0.079148,0.340074,0.909800


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]


TrainOutput(global_step=16876, training_loss=0.18560288980352452, metrics={'train_runtime': 1411.132, 'train_samples_per_second': 478.322, 'train_steps_per_second': 14.949, 'total_flos': 6513410786527440.0, 'train_loss': 0.18560288980352452, 'epoch': 4.0})

In [74]:
# final 실험의 validation 성능을 확인합니다.

final_valid_result = final_trainer.evaluate(valid_dataset)
final_valid_result

Training Loss,Validation Loss,Epoch,Accuracy
0.079148,0.234182,4,0.911667


{'eval_loss': 0.2341821938753128, 'eval_accuracy': 0.9116666666666666}

In [75]:
# final 실험의 test 성능을 확인합니다.
# 목표는 test accuracy 0.91 이상입니다.

final_test_result = final_trainer.evaluate(test_dataset)
final_test_result

Training Loss,Validation Loss,Epoch,Accuracy
0.079148,0.251303,4,0.903974


{'eval_loss': 0.2513033151626587, 'eval_accuracy': 0.9039742384543072}

### STEP 9-3 정리

이번 실험에서는 warmup scheduler, 높은 weight decay, early stopping을 함께 적용했습니다.

확인할 포인트는 다음과 같습니다.

- test accuracy가 0.91 이상으로 올라갔는지
- early stopping이 몇 epoch에서 멈췄는지
- validation accuracy와 test accuracy가 함께 개선되었는지
- validation loss가 안정적인 흐름을 보였는지

만약 0.91을 넘지 못하더라도, `klue/bert-base` 단일 모델에서 test accuracy 0.91 근처가 쉽지 않은 구간임을 확인한 실험으로 기록할 수 있습니다.

### STEP 9-4. 과적합 완화 실험: 보수적 epoch + early stopping

STEP 9-3에서 epoch를 늘렸을 때 train loss는 크게 감소했지만 validation accuracy가 하락하는 과적합 경향이 나타났습니다.

따라서 이번 실험에서는 과한 학습을 줄이는 방향으로 설정을 조정합니다.

조건은 다음과 같습니다.

- learning rate: `2e-5`
- warmup ratio: `0.1`
- scheduler: `linear`
- weight decay: `0.01`
- max epoch: `3`
- early stopping patience: `1`
- bucketing: 적용

목표는 추가 epoch의 기회는 주되, validation 성능이 나빠지면 빠르게 멈추는 것입니다.

In [76]:
# 과적합 완화 실험을 위해 모델을 새로 불러옵니다.
regularized_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2506.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [77]:
# STEP 9-3보다 보수적인 설정입니다.
# weight_decay를 0.01로 되돌리고, max epoch를 3으로 줄이며, patience를 1로 설정합니다.

training_args_regularized = TrainingArguments(
    output_dir='./nsmc_klue_regularized_warmup',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_strategy='steps',
    logging_steps=200,
    report_to='none',
    seed=42,
    train_sampling_strategy='group_by_length'
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [78]:
regularized_trainer = Trainer(
    model=regularized_model,
    args=training_args_regularized,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

regularized_train_result = regularized_trainer.train()
regularized_train_result

Epoch,Training Loss,Validation Loss,Accuracy
1,0.244377,0.229653,0.904933
2,0.190097,0.230291,0.913467
3,0.126302,0.274985,0.912667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]


TrainOutput(global_step=12657, training_loss=0.20737695486763882, metrics={'train_runtime': 1290.7746, 'train_samples_per_second': 313.753, 'train_steps_per_second': 9.806, 'total_flos': 4883816534603100.0, 'train_loss': 0.20737695486763882, 'epoch': 3.0})

In [79]:
# 보수적 실험의 validation 성능을 확인합니다.

regularized_valid_result = regularized_trainer.evaluate(valid_dataset)
regularized_valid_result

Training Loss,Validation Loss,Epoch,Accuracy
0.126302,0.230288,3,0.913467


{'eval_loss': 0.2302878051996231, 'eval_accuracy': 0.9134666666666666}

In [80]:
# 보수적 실험의 test 성능을 확인합니다.

regularized_test_result = regularized_trainer.evaluate(test_dataset)
regularized_test_result

Training Loss,Validation Loss,Epoch,Accuracy
0.126302,0.249678,3,0.906234


{'eval_loss': 0.24967820942401886, 'eval_accuracy': 0.9062343740624438}

### STEP 9-4 정리

이번 실험에서는 STEP 9-3에서 보인 과적합을 줄이기 위해 epoch와 early stopping 조건을 더 보수적으로 설정했습니다.

확인할 포인트는 다음과 같습니다.

- train loss가 지나치게 낮아지기 전에 early stopping이 작동했는지
- validation accuracy가 baseline/bucketing보다 개선되었는지
- test accuracy가 0.91에 가까워졌는지
- validation loss가 STEP 9-3보다 안정적인지

이 실험도 test accuracy 0.91을 넘지 못한다면, `klue/bert-base` 단일 모델과 현재 데이터 구성에서는 0.91 근처가 한계일 수 있다고 해석할 수 있습니다.

## 프로젝트 정리

이번 프로젝트에서는 Hugging Face 실습 흐름을 한국어 NSMC 감성분석 문제에 적용했습니다.

진행한 작업은 다음과 같습니다.

1. NSMC 데이터를 불러오고 train/validation/test 데이터셋을 구성했습니다.
2. `klue/bert-base` tokenizer로 한국어 리뷰 문장을 토큰화했습니다.
3. `AutoModelForSequenceClassification`으로 긍정/부정 분류 모델을 구성했습니다.
4. baseline fine-tuning을 통해 validation accuracy 90% 이상을 달성했습니다.
5. bucketing을 적용한 학습을 수행하고 baseline과 결과를 비교했습니다.
6. learning rate를 낮춘 추가 실험을 통해 성능 개선 가능성을 확인했습니다.
7. warmup scheduler를 적용해 학습 안정화 실험을 진행했습니다.

이를 통해 한국어 BERT 모델을 감성분석 데이터셋에 fine-tuning하는 전체 과정을 확인했습니다.

### 회고 및 추가 개선 방향

초기 실험은 제한된 시간과 GPU 자원을 고려해 2 epoch 기준으로 진행했습니다.

baseline과 bucketing 실험을 먼저 짧게 수행한 뒤, 성능 개선 가능성이 있는 설정을 추가로 실험하는 방식으로 진행했습니다.

warmup scheduler 실험에서는 epoch가 진행되며 validation loss가 안정화되는 경향이 보여, 추가 epoch 학습 시 성능 개선 가능성이 있다고 판단했습니다.

다만 모든 실험을 동일하게 더 긴 epoch로 재학습하기에는 시간이 부족했기 때문에, 이번 프로젝트에서는 baseline, bucketing, learning rate 변경, warmup scheduler 적용 여부에 따른 비교까지만 수행했습니다.

향후에는 다음과 같은 추가 실험을 진행해볼 수 있습니다.

- `EarlyStoppingCallback`을 적용한 3~5 epoch 실험
- validation loss와 accuracy를 함께 고려한 최적 epoch 탐색
- `weight_decay`, `max_length`, batch size 조정 실험
- `beomi/KcELECTRA-base-v2022` 같은 한국어 리뷰/댓글 도메인에 강한 모델과 비교

특히 단순히 epoch를 늘리는 방식보다는, validation loss와 accuracy를 함께 보며 early stopping을 적용하는 것이 더 적절하다고 판단했습니다.